# Tennessee Eastman Process (TEP) Dataset Analysis
## For IndustryFlow IoT Platform - Anomaly Detection

**Dataset:** TEP Faulty Training Data  
**Size:** 5,000,000 samples × 55 features  
**Memory:** 2.0 GB  
**Purpose:** Benchmark anomaly detection algorithms for industrial processes

## 2. Data Structure and Overview

In [11]:
"""
Convert TEP .RData to CSV with 1-second timestamps
For IndustryFlow IoT Platform
"""

import pandas as pd
import pyreadr
from datetime import datetime, timedelta

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_FILE = '../data/TEP_Faulty_Training.RData'
OUTPUT_FILE = '../data/TEP_with_timestamps.csv'
CHUNK_SIZE = 100000  # Process in chunks to manage memory

# Timestamp settings
BASE_TIME = datetime(2024, 1, 1, 0, 0, 0)  # Starting timestamp
INTERVAL_SECONDS = 1  # 1 second between measurements

print("="*70)
print("TEP RData → CSV Conversion with Timestamps")
print("="*70)

# ============================================================================
# STEP 1: Load .RData file
# ============================================================================
print("\n1. Loading .RData file...")
result = pyreadr.read_r(INPUT_FILE)
df = result['faulty_training']  # Correct key for faulty training data

print(f"   ✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# ============================================================================
# STEP 2: Add timestamps (1-second intervals)
# ============================================================================
print("\n2. Adding timestamps (1-second intervals)...")

# Calculate timestamp for each row
# Each simulation starts on a new day, samples are 1 second apart
timestamps = []

for chunk_start in range(0, len(df), CHUNK_SIZE):
    chunk = df.iloc[chunk_start:chunk_start+CHUNK_SIZE]
    
    # Calculate timestamps
    # Days offset based on simulationRun
    sim_offset_days = (chunk['simulationRun'] - 1).astype(int)
    # Seconds offset based on sample
    sample_offset_seconds = (chunk['sample'] - 1).astype(int) * INTERVAL_SECONDS
    
    # Create timestamps
    chunk_timestamps = (
        pd.Timestamp(BASE_TIME) + 
        pd.to_timedelta(sim_offset_days, unit='D') + 
        pd.to_timedelta(sample_offset_seconds, unit='s')
    )
    
    timestamps.extend(chunk_timestamps.tolist())
    
    if (chunk_start + CHUNK_SIZE) % 500000 == 0:
        print(f"   Processed {min(chunk_start+CHUNK_SIZE, len(df)):,} / {len(df):,} rows")

df['timestamp'] = timestamps
print(f"   ✓ Added {len(timestamps):,} timestamps")
print(f"   Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# ============================================================================
# STEP 3: Convert to binary anomaly labels
# ============================================================================
print("\n3. Converting to binary anomaly detection...")

# KEY: faultNumber 0 = normal, 1-20 = anomaly
df['is_anomaly'] = (df['faultNumber'] > 0).astype(int)

print(f"   ✓ Normal (0): {(df['is_anomaly']==0).sum():,} samples ({(df['is_anomaly']==0).mean()*100:.1f}%)")
print(f"   ✓ Anomaly (1): {(df['is_anomaly']==1).sum():,} samples ({(df['is_anomaly']==1).mean()*100:.1f}%)")

# ============================================================================
# STEP 4: Reorder columns
# ============================================================================
print("\n4. Organizing columns...")

# Put important columns first
sensor_cols = [col for col in df.columns if col.startswith('xmeas_') or col.startswith('xmv_')]
metadata_cols = ['timestamp', 'is_anomaly', 'faultNumber', 'simulationRun', 'sample']

df = df[metadata_cols + sensor_cols]
print(f"   ✓ Columns: {len(metadata_cols)} metadata + {len(sensor_cols)} sensors")

# ============================================================================
# STEP 5: Export to CSV (in chunks)
# ============================================================================
print("\n5. Exporting to CSV...")

for i, chunk_start in enumerate(range(0, len(df), CHUNK_SIZE)):
    chunk = df.iloc[chunk_start:chunk_start+CHUNK_SIZE]
    
    if i == 0:
        chunk.to_csv(OUTPUT_FILE, index=False, mode='w')
    else:
        chunk.to_csv(OUTPUT_FILE, index=False, mode='a', header=False)
    
    print(f"   Chunk {i+1} exported ({min(chunk_start+CHUNK_SIZE, len(df)):,} / {len(df):,} rows)")

print(f"\n✓ Export complete!")
print(f"✓ File: {OUTPUT_FILE}")
print(f"✓ Size: ~{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# ============================================================================
# STEP 6: Show sample
# ============================================================================
print("\n6. Data sample:")
print("="*70)
print(df[['timestamp', 'is_anomaly', 'faultNumber', 'xmeas_1', 'xmeas_7', 'xmeas_9']].head(10))

print("\n" + "="*70)
print("✅ CONVERSION COMPLETE!")
print("="*70)
print(f"""
✓ File ready: {OUTPUT_FILE}
✓ Format: CSV with headers
✓ Timestamps: 1-second intervals
✓ Binary labels: is_anomaly (0=normal, 1=anomaly)
✓ Sensors: 52 (41 xmeas + 11 xmv)
✓ Ready for: TimescaleDB, Kafka, Spark Streaming, IndustryFlow
""")

TEP RData → CSV Conversion with Timestamps

1. Loading .RData file...
   ✓ Loaded: 5,000,000 rows × 55 columns
   Memory: 2.01 GB

2. Adding timestamps (1-second intervals)...
   Processed 500,000 / 5,000,000 rows
   Processed 1,000,000 / 5,000,000 rows
   Processed 1,500,000 / 5,000,000 rows
   Processed 2,000,000 / 5,000,000 rows
   Processed 2,500,000 / 5,000,000 rows
   Processed 3,000,000 / 5,000,000 rows
   Processed 3,500,000 / 5,000,000 rows
   Processed 4,000,000 / 5,000,000 rows
   Processed 4,500,000 / 5,000,000 rows
   Processed 5,000,000 / 5,000,000 rows
   ✓ Added 5,000,000 timestamps
   Date range: 2024-01-01 00:00:00 to 2025-05-14 00:08:19

3. Converting to binary anomaly detection...
   ✓ Normal (0): 0 samples (0.0%)
   ✓ Anomaly (1): 5,000,000 samples (100.0%)

4. Organizing columns...
   ✓ Columns: 5 metadata + 52 sensors

5. Exporting to CSV...
   Chunk 1 exported (100,000 / 5,000,000 rows)
   Chunk 2 exported (200,000 / 5,000,000 rows)
   Chunk 3 exported (300,000 

KeyboardInterrupt: 